In [1]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd

from Bio.PDB import PDBParser, MMCIFParser
from Bio.PDB.Polypeptide import is_aa
from Bio.PDB.vectors import calc_dihedral

In [2]:
DATA_DIR = Path("data")

PDB_FILES_DIR = DATA_DIR / "raw" / "pdb_files"
CIF_FILES_DIR = DATA_DIR / "raw"/ "cif_files"

PDB_IDS_JSON_PATH = DATA_DIR / "raw" / "filtered_pdb_ids.json"

PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

N_BINS = 36
BIN_SIZE_DEGREES = 10

IGNORE_INDEX = -100

In [3]:
def load_pdb_ids(json_path: Path):
    with open(json_path, "r") as f:
        pdb_ids = json.load(f)

    pdb_ids = [pdb_id.upper() for pdb_id in pdb_ids]

    return pdb_ids

In [4]:
pdb_ids = load_pdb_ids(PDB_IDS_JSON_PATH)

print("Number of PDB IDs:", len(pdb_ids))
print(pdb_ids[:10])

Number of PDB IDs: 6108
['2BO5', '2BW2', '2IVW', '4ME2', '4MEI', '4MFI', '4MFJ', '4MGS', '4MHP', '4MPL']


In [5]:
def find_structure_file(pdb_id: str):
    pdb_id_lower = pdb_id.lower()

    candidate_paths = [
        CIF_FILES_DIR / f"{pdb_id_lower}.cif",
        PDB_FILES_DIR / f"pdb{pdb_id_lower}.ent",
    ]

    for path in candidate_paths:
        if path.exists():
            return path

    return None

In [6]:
def load_structure(pdb_id: str, structure_path: Path):
    suffix = structure_path.suffix.lower()

    if suffix in {".cif"}:
        parser = MMCIFParser(QUIET=True)
    elif suffix in {".ent"}:
        parser = PDBParser(QUIET=True)
    else:
        raise ValueError(f"Unsupported structure format: {structure_path}")

    return parser.get_structure(pdb_id, structure_path)

In [7]:
STANDARD_AA_3 = {
    "ALA", "ARG", "ASN", "ASP", "CYS",
    "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO",
    "SER", "THR", "TRP", "TYR", "VAL",
}

In [8]:
CHI1_ATOMS = {
    "ARG": ("N", "CA", "CB", "CG"),
    "ASN": ("N", "CA", "CB", "CG"),
    "ASP": ("N", "CA", "CB", "CG"),
    "CYS": ("N", "CA", "CB", "SG"),
    "GLN": ("N", "CA", "CB", "CG"),
    "GLU": ("N", "CA", "CB", "CG"),
    "HIS": ("N", "CA", "CB", "CG"),
    "ILE": ("N", "CA", "CB", "CG1"),
    "LEU": ("N", "CA", "CB", "CG"),
    "LYS": ("N", "CA", "CB", "CG"),
    "MET": ("N", "CA", "CB", "CG"),
    "PHE": ("N", "CA", "CB", "CG"),
    "PRO": ("N", "CA", "CB", "CG"),
    "SER": ("N", "CA", "CB", "OG"),
    "THR": ("N", "CA", "CB", "OG1"),
    "TRP": ("N", "CA", "CB", "CG"),
    "TYR": ("N", "CA", "CB", "CG"),
    "VAL": ("N", "CA", "CB", "CG1"),
}

In [9]:
def normalize_angle_degrees(angle):
    if angle is None or pd.isna(angle):
        return np.nan

    return float(angle) % 360.0

In [10]:
def angle_to_10deg_bin(angle):
    angle = normalize_angle_degrees(angle)

    if pd.isna(angle):
        return IGNORE_INDEX

    return int(angle // BIN_SIZE_DEGREES)

In [11]:
def get_atom_vector(residue, atom_name):
    if atom_name not in residue:
        return None

    return residue[atom_name].get_vector()

def compute_dihedral_degrees(residue, atom_names):
    vectors = []

    for atom_name in atom_names:
        vector = get_atom_vector(residue, atom_name)

        if vector is None:
            return np.nan

        vectors.append(vector)

    angle_rad = calc_dihedral(*vectors)
    angle_deg = math.degrees(angle_rad)

    return normalize_angle_degrees(angle_deg)

In [12]:
def get_atom_coord(residue, atom_name):
    if atom_name not in residue:
        return None

    return residue[atom_name].get_coord()

def compute_phi(prev_residue, residue):
    if prev_residue is None:
        return np.nan

    c_prev = get_atom_vector(prev_residue, "C")
    n = get_atom_vector(residue, "N")
    ca = get_atom_vector(residue, "CA")
    c = get_atom_vector(residue, "C")

    if c_prev is None or n is None or ca is None or c is None:
        return np.nan

    angle_rad = calc_dihedral(c_prev, n, ca, c)
    angle_deg = math.degrees(angle_rad)

    return normalize_angle_degrees(angle_deg)

def compute_psi(residue, next_residue):
    if next_residue is None:
        return np.nan

    n = get_atom_vector(residue, "N")
    ca = get_atom_vector(residue, "CA")
    c = get_atom_vector(residue, "C")
    n_next = get_atom_vector(next_residue, "N")

    if n is None or ca is None or c is None or n_next is None:
        return np.nan

    angle_rad = calc_dihedral(n, ca, c, n_next)
    angle_deg = math.degrees(angle_rad)

    return normalize_angle_degrees(angle_deg)

def angle_to_sin_cos(angle):
    if pd.isna(angle):
        return 0.0, 0.0

    angle_rad = math.radians(float(angle))

    return math.sin(angle_rad), math.cos(angle_rad)

In [13]:
def parse_residue_id(residue):
    hetero_flag, residue_number, insertion_code = residue.id

    insertion_code = insertion_code.strip()

    return hetero_flag, residue_number, insertion_code

In [14]:
def extract_residues_from_structure(pdb_id: str, structure_path: Path):
    structure = load_structure(pdb_id, structure_path)

    rows = []

    model = structure[0]

    for chain in model:
        chain_id = chain.id

        valid_residues = []

        for residue in chain:
            hetero_flag, residue_number, insertion_code = parse_residue_id(residue)

            if hetero_flag.strip() != "":
                continue

            resname = residue.get_resname().strip().upper()

            if resname not in STANDARD_AA_3:
                continue

            if not is_aa(residue, standard=True):
                continue

            valid_residues.append(residue)

        for i, residue in enumerate(valid_residues):
            prev_residue = valid_residues[i - 1] if i > 0 else None
            next_residue = valid_residues[i + 1] if i + 1 < len(valid_residues) else None

            hetero_flag, residue_number, insertion_code = parse_residue_id(residue)
            resname = residue.get_resname().strip().upper()

            chi1_angle = np.nan

            if resname in CHI1_ATOMS:
                chi1_angle = compute_dihedral_degrees(
                    residue=residue,
                    atom_names=CHI1_ATOMS[resname],
                )

            if pd.isna(chi1_angle):
                chi1_bin = IGNORE_INDEX
                target_mask = 0
            else:
                chi1_bin = angle_to_10deg_bin(chi1_angle)
                target_mask = 1

            phi_angle = compute_phi(prev_residue, residue)
            psi_angle = compute_psi(residue, next_residue)

            phi_sin, phi_cos = angle_to_sin_cos(phi_angle)
            psi_sin, psi_cos = angle_to_sin_cos(psi_angle)

            ca_coord = get_atom_coord(residue, "CA")

            if ca_coord is None:
                ca_x = np.nan
                ca_y = np.nan
                ca_z = np.nan
                has_ca = 0
            else:
                ca_x = float(ca_coord[0])
                ca_y = float(ca_coord[1])
                ca_z = float(ca_coord[2])
                has_ca = 1

            rows.append({
                "pdb_id": pdb_id,
                "chain_id": chain_id,
                "residue_number": residue_number,
                "insertion_code": insertion_code,
                "resname": resname,

                "chi1_angle": chi1_angle,
                "chi1_bin_10deg": chi1_bin,
                "target_mask": target_mask,

                "phi_angle": phi_angle,
                "psi_angle": psi_angle,
                "phi_sin": phi_sin,
                "phi_cos": phi_cos,
                "psi_sin": psi_sin,
                "psi_cos": psi_cos,
                "has_phi": int(not pd.isna(phi_angle)),
                "has_psi": int(not pd.isna(psi_angle)),

                "ca_x": ca_x,
                "ca_y": ca_y,
                "ca_z": ca_z,
                "has_ca": has_ca,
            })

    return pd.DataFrame(rows)

In [15]:
AA_PROPERTIES = {
    "ALA": {
        "polarity_group": "nonpolar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "small",
        "chemical_group": "aliphatic",
        "physicochemical_group": "nonpolar_aliphatic",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "none",
    },
    "ARG": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophilic",
        "volume_group": "large",
        "chemical_group": "basic",
        "physicochemical_group": "positive",
        "charge_group": "positive",
        "hydrogen_donor_acceptor_group": "donor",
    },
    "ASN": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophilic",
        "volume_group": "medium",
        "chemical_group": "amide",
        "physicochemical_group": "polar_uncharged",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "donor_acceptor",
    },
    "ASP": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophilic",
        "volume_group": "medium",
        "chemical_group": "acidic",
        "physicochemical_group": "negative",
        "charge_group": "negative",
        "hydrogen_donor_acceptor_group": "acceptor",
    },
    "CYS": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "small",
        "chemical_group": "sulfur",
        "physicochemical_group": "special",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "donor",
    },
    "GLN": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophilic",
        "volume_group": "large",
        "chemical_group": "amide",
        "physicochemical_group": "polar_uncharged",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "donor_acceptor",
    },
    "GLU": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophilic",
        "volume_group": "large",
        "chemical_group": "acidic",
        "physicochemical_group": "negative",
        "charge_group": "negative",
        "hydrogen_donor_acceptor_group": "acceptor",
    },
    "GLY": {
        "polarity_group": "nonpolar",
        "hydropathy_group": "neutral",
        "volume_group": "very_small",
        "chemical_group": "special",
        "physicochemical_group": "special",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "none",
    },
    "HIS": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophilic",
        "volume_group": "large",
        "chemical_group": "basic_aromatic",
        "physicochemical_group": "positive_weak",
        "charge_group": "positive_weak",
        "hydrogen_donor_acceptor_group": "donor_acceptor",
    },
    "ILE": {
        "polarity_group": "nonpolar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "large",
        "chemical_group": "aliphatic",
        "physicochemical_group": "nonpolar_aliphatic",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "none",
    },
    "LEU": {
        "polarity_group": "nonpolar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "large",
        "chemical_group": "aliphatic",
        "physicochemical_group": "nonpolar_aliphatic",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "none",
    },
    "LYS": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophilic",
        "volume_group": "large",
        "chemical_group": "basic",
        "physicochemical_group": "positive",
        "charge_group": "positive",
        "hydrogen_donor_acceptor_group": "donor",
    },
    "MET": {
        "polarity_group": "nonpolar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "large",
        "chemical_group": "sulfur",
        "physicochemical_group": "nonpolar_sulfur",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "none",
    },
    "PHE": {
        "polarity_group": "nonpolar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "large",
        "chemical_group": "aromatic",
        "physicochemical_group": "aromatic",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "none",
    },
    "PRO": {
        "polarity_group": "nonpolar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "medium",
        "chemical_group": "cyclic",
        "physicochemical_group": "special",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "none",
    },
    "SER": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophilic",
        "volume_group": "small",
        "chemical_group": "hydroxyl",
        "physicochemical_group": "polar_uncharged",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "donor_acceptor",
    },
    "THR": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophilic",
        "volume_group": "medium",
        "chemical_group": "hydroxyl",
        "physicochemical_group": "polar_uncharged",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "donor_acceptor",
    },
    "TRP": {
        "polarity_group": "nonpolar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "large",
        "chemical_group": "aromatic",
        "physicochemical_group": "aromatic",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "donor",
    },
    "TYR": {
        "polarity_group": "polar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "large",
        "chemical_group": "aromatic_hydroxyl",
        "physicochemical_group": "aromatic_polar",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "donor_acceptor",
    },
    "VAL": {
        "polarity_group": "nonpolar",
        "hydropathy_group": "hydrophobic",
        "volume_group": "medium",
        "chemical_group": "aliphatic",
        "physicochemical_group": "nonpolar_aliphatic",
        "charge_group": "neutral",
        "hydrogen_donor_acceptor_group": "none",
    },
}

PROPERTY_COLUMNS = [
    "polarity_group",
    "hydropathy_group",
    "volume_group",
    "chemical_group",
    "physicochemical_group",
    "charge_group",
    "hydrogen_donor_acceptor_group",
]

In [16]:
def add_amino_acid_properties(df: pd.DataFrame):
    df = df.copy()

    for column in PROPERTY_COLUMNS:
        df[column] = df["resname"].map(
            lambda aa: AA_PROPERTIES[aa][column]
        )

    return df

In [17]:
def collect_residue_dataset(pdb_ids):
    all_dfs = []
    missing_files = []
    failed_structures = []

    for i, pdb_id in enumerate(pdb_ids):
        structure_path = find_structure_file(pdb_id)

        if structure_path is None:
            missing_files.append(pdb_id)
            continue

        try:
            df_one = extract_residues_from_structure(
                pdb_id=pdb_id,
                structure_path=structure_path,
            )

            if len(df_one) == 0:
                failed_structures.append((pdb_id, "empty dataframe"))
                continue

            df_one = add_amino_acid_properties(df_one)
            all_dfs.append(df_one)

        except Exception as e:
            failed_structures.append((pdb_id, repr(e)))

        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(pdb_ids)}")

    if len(all_dfs) == 0:
        raise RuntimeError("No structures were successfully processed.")

    dataset_df = pd.concat(all_dfs, ignore_index=True)

    return dataset_df, missing_files, failed_structures

In [18]:
residue_df, missing_files, failed_structures = collect_residue_dataset(pdb_ids)

Processed 100/6108
Processed 200/6108
Processed 300/6108
Processed 400/6108
Processed 500/6108
Processed 600/6108
Processed 700/6108
Processed 800/6108
Processed 900/6108
Processed 1000/6108
Processed 1100/6108
Processed 1200/6108
Processed 1300/6108
Processed 1400/6108
Processed 1500/6108
Processed 1600/6108
Processed 1700/6108
Processed 1800/6108
Processed 1900/6108
Processed 2000/6108
Processed 2100/6108
Processed 2200/6108
Processed 2300/6108
Processed 2400/6108
Processed 2500/6108
Processed 2600/6108
Processed 2700/6108
Processed 2800/6108
Processed 2900/6108
Processed 3000/6108
Processed 3100/6108
Processed 3200/6108
Processed 3300/6108
Processed 3400/6108
Processed 3500/6108
Processed 3600/6108
Processed 3700/6108
Processed 3800/6108
Processed 3900/6108
Processed 4000/6108
Processed 4100/6108
Processed 4200/6108
Processed 4300/6108
Processed 4400/6108
Processed 4500/6108
Processed 4600/6108
Processed 4700/6108
Processed 4800/6108
Processed 4900/6108
Processed 5000/6108
Processed

In [19]:
residue_df[
    [
        "pdb_id",
        "chain_id",
        "residue_number",
        "resname",
        "chi1_angle",
        "chi1_bin_10deg",
        "target_mask",
        "phi_angle",
        "psi_angle",
        "phi_sin",
        "phi_cos",
        "psi_sin",
        "psi_cos",
        "has_phi",
        "has_psi",
        "ca_x",
        "ca_y",
        "ca_z",
        "has_ca",
    ]
].head()

,pdb_id,chain_id,residue_number,resname,chi1_angle,chi1_bin_10deg,target_mask,phi_angle,psi_angle,phi_sin,phi_cos,psi_sin,psi_cos,has_phi,has_psi,ca_x,ca_y,ca_z,has_ca
0,2BO5,A,1,PHE,267.667646,26,1,NaN,181.918425,0.000000,0.000000,-0.033477,-0.999440,0,1,24.882000,12.444,24.544001,1
1,2BO5,A,2,ALA,NaN,-100,0,60.562806,33.116558,0.870895,0.491469,0.546344,0.837561,1,1,22.478001,11.484,21.757999,1
2,2BO5,A,3,LYS,270.463347,27,1,299.348529,285.420907,-0.871654,0.490121,-0.963998,0.265908,1,1,20.426001,9.539,24.302000,1
3,2BO5,A,4,LEU,314.589093,31,1,61.046673,156.017583,0.875014,0.484097,0.406456,-0.913670,1,1,16.906000,9.970,22.927000,1
4,2BO5,A,5,VAL,286.087049,28,1,248.450014,326.412570,-0.930097,-0.367313,-0.553209,0.833043,1,1,14.043000,7.605,23.750000,1


In [20]:
print("has_phi:")
print(residue_df["has_phi"].value_counts())

print("\nhas_psi:")
print(residue_df["has_psi"].value_counts())

print("\nhas_ca:")
print(residue_df["has_ca"].value_counts())

print("\ntarget_mask:")
print(residue_df["target_mask"].value_counts())

has_phi:
has_phi
1    1156898
0       8775
Name: count, dtype: int64

has_psi:
has_psi
1    1156895
0       8778
Name: count, dtype: int64

has_ca:
has_ca
1    1165673
Name: count, dtype: int64

target_mask:
target_mask
1    973654
0    192019
Name: count, dtype: int64


In [21]:
print("Missing files:", len(missing_files))
print("Failed structures:", len(failed_structures))

print("First missing files:", missing_files[:10])
print("First failed structures:", failed_structures[:10])

Missing files: 0
Failed structures: 0
First missing files: []
First failed structures: []


In [22]:
# output_path = PROCESSED_DIR / "chi1_residue_level_dataset.csv"

# residue_df.to_csv(output_path, index=False)

# print("Saved to:", output_path)

output_path = PROCESSED_DIR / "chi1_residue_level_dataset_with_backbone.csv"

residue_df.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: data/processed/chi1_residue_level_dataset_with_backbone.csv
